### «Умный помощник» для оформления командировок

#### Установка зависимостей

In [1]:
%pip install -q langchain langgraph langchain-community chromadb sentence-transformers torch pandas openai python-dotenv langchain-openai

Note: you may need to restart the kernel to use updated packages.


#### Конфигурация

In [3]:
import os
from typing import Dict, List, Any
import pandas as pd
from io import StringIO
import traceback
from dotenv import load_dotenv

# Загружаем .env
load_dotenv()

# === Чтение конфигурации ===
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "openrouter").lower()

# OpenRouter
OPENROUTER_API_KEY = os.getenv("API_KEY", "")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-20b:free")

# Локальная (Ollama / LM Studio)
# LOCAL_MODEL_NAME = os.getenv("LOCAL_MODEL_NAME", "llama3.2")
# LOCAL_BASE_URL = os.getenv("LOCAL_BASE_URL", "http://localhost:11434")
# LOCAL_API_KEY = os.getenv("LOCAL_API_KEY", "ollama")

# Эмбеддинги
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")

# Логирование
LOG_LEVEL = os.getenv("LOG_LEVEL", "info").upper()

# === Инициализация LLM ===
llm = None

if LLM_PROVIDER == "openrouter" and OPENROUTER_API_KEY:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url=OPENROUTER_BASE_URL,
        model=OPENROUTER_MODEL,
        temperature=0.7
    )
    print(f"✅ LLM инициализирован: OpenRouter / {OPENROUTER_MODEL}")
    # Быстрый пинг
    try:
        response = llm.invoke("Ответь OK")
        print(f"✅ Пинг успешен: {response.content[:50]}")
    except Exception as e:
        print(f"❌ Пинг провален: {e}")
        
# elif LLM_PROVIDER == "local":
#     from langchain_openai import ChatOpenAI
#     llm = ChatOpenAI(
#         api_key=LOCAL_API_KEY,
#         base_url=LOCAL_BASE_URL,
#         model=LOCAL_MODEL_NAME,
#         temperature=0.7
#     )
#     print(f"✅ LLM инициализирован: локальная модель {LOCAL_MODEL_NAME} ({LOCAL_BASE_URL})")
#     # Быстрый пинг
#     try:
#         response = llm.invoke("Ответь OK")
#         print(f"✅ Пинг успешен: {response.content[:50]}")
#     except Exception as e:
#         print(f"❌ Пинг провален: {e}")
        
else:
    print("⚠️ LLM не сконфигурирован (нет API ключа или выбран неподдерживаемый провайдер).")
    print("   Будет использована заглушка DummyLLM, которая не генерирует осмысленные ответы.")
    class DummyLLM:
        def invoke(self, prompt, **kwargs):
            return f"[DummyLLM] Нет реального LLM. Промпт: {prompt[:200]}..."
    llm = DummyLLM()

def print_config():
    print("\n" + "="*50)
    print("ТЕКУЩАЯ КОНФИГУРАЦИЯ")
    print("="*50)
    print(f"LLM Provider: {LLM_PROVIDER.upper()}")
    if LLM_PROVIDER == "openrouter":
        print(f"Model: {OPENROUTER_MODEL}")
        print(f"API Key: {'***' if OPENROUTER_API_KEY else 'НЕТ'}")
    # elif LLM_PROVIDER == "local":
    #     print(f"Model: {LOCAL_MODEL_NAME}")
    #     print(f"Base URL: {LOCAL_BASE_URL}")
    print(f"Embedding: {EMBEDDING_MODEL}")
    print("="*50 + "\n")

print_config()

c:\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LLM инициализирован: OpenRouter / openai/gpt-oss-20b:free
✅ Пинг успешен: OK

ТЕКУЩАЯ КОНФИГУРАЦИЯ
LLM Provider: OPENROUTER
Model: openai/gpt-oss-20b:free
API Key: ***
Embedding: sentence-transformers/all-MiniLM-L6-v2



#### из файлов

In [7]:
# Загрузка CSV
tickets_itab = pd.read_csv("data/flights.csv")
print(f"✅ Загружено {len(tickets_itab)} рейсов из data/flights.csv")
print(tickets_itab.head())

# Загрузка политики из data/policy.txt
with open("data/policy.txt", "r", encoding="utf-8") as f:
    policy_text = f.read()
print(f"✅ Загружен текст политики ({len(policy_text)} символов) из data/policy.txt")
print(policy_text[:500] + "..." if len(policy_text) > 500 else policy_text)

✅ Загружено 300 рейсов из data/flights.csv
  line;flight;departure;arrival;departure_date;price
0      Россия;РО5091;Иркутск;Москва;2026-06-05;15544
1  ЮТэйр;ЮТ4166;Санкт-Петербург;Воркута;2026-01-0...
2  Red Wings;RE8185;Волгоград;Иркутск;2026-08-18;...
3  Россия;РО5925;Москва;Санкт-Петербург;2026-01-1...
4    Победа;ПО3052;Волгоград;Иркутск;2026-10-24;8009
✅ Загружен текст политики (5852 символов) из data/policy.txt
КОРПОРАТИВНАЯ ПОЛИТИКА КОМАНДИРОВОК ООО "Ромашка"

Версия 2.4, вступает в силу с 01.06.2026

1. ОБЩИЕ ПОЛОЖЕНИЯ

Настоящая политика определяет правила направления сотрудников в служебные командировки, условия оплаты проезда, проживания, суточных, а также порядок отчётности. Политика обязательна для всех подразделений.

2. АВИАБИЛЕТЫ: МАКСИМАЛЬНАЯ ЦЕНА И КАТЕГОРИИ ПО ДОЛЖНОСТЯМ

2.1. Максимальная цена билета (туда-обратно, эконом-класс, включая сборы) не может превышать:
    - Для перелётов внутр...


#### Embeddings